# Vessel Corrosion Detection — YOLOv8l Training
**Full workflow:** Setup → Dataset check → Tuning → Train → Validate → Test → Results

---
**Before running:**
1. Upload `training_colab_upload.zip` to `My Drive/colab/` in Google Drive
2. Runtime → Change runtime type → **GPU** (T4 or A100)
3. Run cells **one by one** in order

---
**Workflow options:**
- **With tuning (recommended):** STEP 1 → 2 → 3 → 4 → 5 → 6 → 8 → 9 → 10 → 11
- **Without tuning (faster):**   STEP 1 → 2 → 3 → 4 → 7 → 8 → 9 → 10 → 11

In [ ]:
# ── STEP 1: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted at /content/drive")

In [ ]:
# ── STEP 2: Check GPU ─────────────────────────────────────────────────────────
!nvidia-smi
import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── STEP 3: Extract zip and install requirements ───────────────────────────────
import os

ZIP_PATH    = "/content/drive/MyDrive/colab/training_colab_upload.zip"
EXTRACT_DIR = "/content"
WORK_DIR    = "/content/training_colab_upload"

# Remove old run if re-running this cell
if os.path.exists(WORK_DIR):
    !rm -rf {WORK_DIR}

# Prefer zip (faster than copying thousands of files from Drive)
if os.path.exists(ZIP_PATH):
    print("Extracting from zip...")
    !unzip -q {ZIP_PATH} -d {EXTRACT_DIR}
else:
    print("Zip not found — copying folder directly from Drive...")
    !cp -R "/content/drive/MyDrive/colab/training_colab_upload" {EXTRACT_DIR}

# Set working directory for all subsequent cells
os.chdir(WORK_DIR)
print(f"Working directory: {os.getcwd()}")

# Install dependencies
print("\nInstalling requirements...")
!pip install -q -r requirements.txt
print("Requirements installed.")

In [ ]:
# ── STEP 4: Verify dataset ────────────────────────────────────────────────────
import os

splits = ["train", "val", "test"]
print("=== Dataset Check: data/yolo_640 ===\n")

total_images = 0
for split in splits:
    img_dir = f"data/yolo_640/images/{split}"
    lbl_dir = f"data/yolo_640/labels/{split}"

    imgs   = [f for f in os.listdir(img_dir) if not f.startswith('.')]
    labels = [f for f in os.listdir(lbl_dir) if not f.startswith('.')]
    empty  = sum(1 for f in labels if os.path.getsize(os.path.join(lbl_dir, f)) == 0)

    total_images += len(imgs)
    print(f"  {split:5s}: {len(imgs):4d} images | {len(labels):4d} labels | {empty:3d} background (empty labels)")

print(f"\n  Total: {total_images} tiles across all splits")
print("\nDataset looks good — ready to train." if total_images > 0 else "\nERROR: No tiles found. Check extraction.")

---
## Hyperparameter Tuning (STEP 5 & 6)
STEP 5 mencari kombinasi hyperparameter terbaik melalui **Random Search** (15 percobaan × 25 epoch).
STEP 6 menggunakan hasil STEP 5 untuk training penuh 200 epoch.

**Estimasi waktu STEP 5:**
- T4 GPU  : ~8–12 menit per trial → 15 trial ≈ **2–3 jam**
- A100    : ~4–6 menit per trial → 15 trial ≈ **1–1.5 jam**

> Lewati STEP 5 & 6 dan langsung ke **STEP 7** jika ingin training tanpa tuning.

In [ ]:
# ── STEP 5: Hyperparameter Tuning (Random Search, in-process) ─────────────────
import os, yaml, random, gc
import torch
from ultralytics import YOLO

TUNE_OUTPUT = "/content/drive/MyDrive/colab/vessel-corrosion-runs/tune"
os.makedirs(TUNE_OUTPUT, exist_ok=True)

N_TRIALS = 15

SEARCH_SPACE = {
    "lr0":        (0.0005, 0.008),
    "lrf":        (0.005,  0.05),
    "box":        (4.0,    12.0),
    "cls":        (0.2,    0.8),
    "copy_paste": (0.0,    0.3),
    "mixup":      (0.0,    0.15),
    "degrees":    (0.0,    20.0),
    "hsv_s":      (0.3,    0.8),
    "hsv_v":      (0.2,    0.6),
}

FIXED = dict(
    data         = "configs/data.yaml",
    epochs       = 25,
    imgsz        = 640,
    batch        = 16,
    device       = 0,
    optimizer    = "AdamW",
    patience     = 25,
    cache        = "ram",
    cos_lr       = True,
    nbs          = 64,
    close_mosaic = 5,
    warmup_epochs= 3.0,
    hsv_h        = 0.015,
    translate    = 0.10,
    scale        = 0.50,
    shear        = 3.0,
    perspective  = 0.0003,
    flipud       = 0.3,
    fliplr       = 0.5,
    mosaic       = 1.0,
    plots        = False,
    save         = False,
    val          = True,
    verbose      = False,
    seed         = 42,
    workers      = 4,
)

results_log = []
best_map50  = 0.0
best_params = {}

print(f"{'Trial':>6} | {'lr0':>8} | {'box':>6} | {'cls':>5} | {'copy_paste':>10} | {'mAP50':>7} | {'P':>7} | {'R':>7}")
print("-" * 80)

for trial in range(1, N_TRIALS + 1):
    sampled = {k: round(random.uniform(lo, hi), 5)
               for k, (lo, hi) in SEARCH_SPACE.items()}

    model   = YOLO("yolo11l.pt")
    metrics = model.train(**FIXED, **sampled)

    map50 = float(metrics.results_dict.get("metrics/mAP50(B)",     0.0))
    prec  = float(metrics.results_dict.get("metrics/precision(B)", 0.0))
    rec   = float(metrics.results_dict.get("metrics/recall(B)",    0.0))

    print(f"{trial:>6} | {sampled['lr0']:>8.5f} | {sampled['box']:>6.2f} | "
          f"{sampled['cls']:>5.2f} | {sampled['copy_paste']:>10.2f} | "
          f"{map50:>7.4f} | {prec:>7.4f} | {rec:>7.4f}")

    results_log.append({"trial": trial, "mAP50": map50, "precision": prec,
                         "recall": rec, **sampled})

    if map50 > best_map50:
        best_map50  = map50
        best_params = sampled.copy()

    del model
    gc.collect()
    torch.cuda.empty_cache()

output = {"best_mAP50": best_map50, "best_hyperparameters": best_params,
          "all_trials": results_log}

RESULT_YAML = f"{TUNE_OUTPUT}/tune_results.yaml"
with open(RESULT_YAML, "w") as f:
    yaml.dump(output, f, default_flow_style=False)

print(f"\n{'='*80}")
print(f"Tuning selesai! Best mAP50 = {best_map50:.4f}")
for k, v in best_params.items():
    print(f"  {k}: {v}")
print(f"\n  Hasil lengkap: {RESULT_YAML}")

In [ ]:
# ── STEP 6: Training Penuh dengan Hyperparameter Terbaik ──────────────────────
import yaml
from ultralytics import YOLO

TUNE_OUTPUT = "/content/drive/MyDrive/colab/vessel-corrosion-runs/tune"
RESULT_YAML = f"{TUNE_OUTPUT}/tune_results.yaml"
OUTPUT_DIR  = "/content/drive/MyDrive/colab/vessel-corrosion-runs/detect"
RUN_NAME    = "ship_corrosion_640_yolo11l_tuned"

with open(RESULT_YAML) as f:
    tune_data = yaml.safe_load(f)

best_hp  = tune_data["best_hyperparameters"]
best_map = tune_data["best_mAP50"]

print(f"=== Hyperparameter Terbaik (dari {len(tune_data['all_trials'])} trial) ===")
print(f"  Best mAP50 saat tuning: {best_map:.4f}")
for k, v in best_hp.items():
    print(f"  {k}: {v}")
print()

model = YOLO("yolo11l.pt")
model.train(
    data         = "configs/data.yaml",
    epochs       = 200,
    imgsz        = 640,
    batch        = 16,
    device       = 0,
    optimizer    = "AdamW",
    patience     = 50,
    cache        = "ram",
    cos_lr       = True,
    nbs          = 64,
    close_mosaic = 10,
    warmup_epochs= 3.0,
    pretrained   = True,
    plots        = True,
    seed         = 42,
    workers      = 4,
    project      = OUTPUT_DIR,
    name         = RUN_NAME,
    hsv_h        = 0.015,
    translate    = 0.10,
    scale        = 0.50,
    shear        = 3.0,
    perspective  = 0.0003,
    flipud       = 0.3,
    fliplr       = 0.5,
    mosaic       = 1.0,
    **best_hp,
)
print(f"\nTraining selesai! Lanjut ke STEP 8 & 9 untuk evaluasi.")

---
## Training Tanpa Tuning (STEP 7)
Lewati STEP 7 jika sudah menjalankan STEP 5 & 6.

Gunakan STEP 7 jika:
- Tidak ada waktu untuk tuning, atau
- Ingin training ulang dengan hyperparameter manual

In [ ]:
# ── STEP 7: Training Langsung (tanpa tuning) ───────────────────────────────────
RUN_NAME   = "ship_corrosion_640_yolo11l"
OUTPUT_DIR = "/content/drive/MyDrive/colab/vessel-corrosion-runs/detect"

!python scripts/train.py \
  --data     configs/data.yaml \
  --model    yolo11l.pt \
  --epochs   200 \
  --imgsz    640 \
  --batch    16 \
  --lr0      0.002 \
  --optimizer AdamW \
  --patience  50 \
  --device    0 \
  --light-augment \
  --project  {OUTPUT_DIR} \
  --name     {RUN_NAME}

# ── Resume setelah disconnect ──────────────────────────────────────────────────
# LAST_PT = f"{OUTPUT_DIR}/{RUN_NAME}/weights/last.pt"
# !python scripts/train.py \
#   --data     configs/data.yaml \
#   --model    {LAST_PT} \
#   --epochs   200 \
#   --imgsz    640 \
#   --batch    16 \
#   --lr0      0.002 \
#   --optimizer AdamW \
#   --patience  50 \
#   --device    0 \
#   --light-augment \
#   --project  {OUTPUT_DIR} \
#   --name     {RUN_NAME}

---
## Evaluasi & Hasil
Sesuaikan `RUN_NAME` di STEP 8–11 dengan nama run yang dipakai:
- Jika pakai STEP 6 (dengan tuning)  → `ship_corrosion_640_yolov8l_tuned`
- Jika pakai STEP 7 (tanpa tuning)   → `ship_corrosion_640_yolov8l`

In [ ]:
# ── STEP 8: Evaluasi Validation Set ──────────────────────────────────────────
# conf=0.001 menghitung full mAP sweep — cara yang benar untuk mengukur mAP.
# Hasil (plots + metrics) disimpan ke MyDrive/colab/vessel-corrosion-runs/eval/

# Ganti sesuai run yang dipakai (STEP 6 atau STEP 7)
RUN_NAME   = "ship_corrosion_640_yolov8l_tuned"   # ganti ke ship_corrosion_640_yolov8l jika pakai STEP 7
OUTPUT_DIR = "/content/drive/MyDrive/colab/vessel-corrosion-runs"
BEST_PT    = f"{OUTPUT_DIR}/detect/{RUN_NAME}/weights/best.pt"

print("=== Validation Set ===")
!python scripts/evaluate.py \
  --weights {BEST_PT} \
  --data    configs/data.yaml \
  --imgsz   640 \
  --batch   8 \
  --split   val \
  --conf    0.001 \
  --iou     0.6 \
  --device  0 \
  --project {OUTPUT_DIR}/eval \
  --name    {RUN_NAME}_val

In [ ]:
# ── STEP 9: Evaluasi Test Set ─────────────────────────────────────────────────
# Skor akhir yang tidak bias — jalankan SEKALI setelah selesai tuning.
# Hasil disimpan ke MyDrive/colab/vessel-corrosion-runs/eval/

RUN_NAME   = "ship_corrosion_640_yolov8l_tuned"   # ganti ke ship_corrosion_640_yolov8l jika pakai STEP 7
OUTPUT_DIR = "/content/drive/MyDrive/colab/vessel-corrosion-runs"
BEST_PT    = f"{OUTPUT_DIR}/detect/{RUN_NAME}/weights/best.pt"

print("=== Test Set ===")
!python scripts/evaluate.py \
  --weights {BEST_PT} \
  --data    configs/data.yaml \
  --imgsz   640 \
  --batch   8 \
  --split   test \
  --conf    0.001 \
  --iou     0.6 \
  --device  0 \
  --project {OUTPUT_DIR}/eval \
  --name    {RUN_NAME}_test

In [ ]:
# ── STEP 10: Tampilkan Training Curves ────────────────────────────────────────
import os
from IPython.display import Image, display

RUN_NAME   = "ship_corrosion_640_yolov8l_tuned"   # ganti ke ship_corrosion_640_yolov8l jika pakai STEP 7
OUTPUT_DIR = "/content/drive/MyDrive/colab/vessel-corrosion-runs"
RUN_DIR    = f"{OUTPUT_DIR}/detect/{RUN_NAME}"

plots = [
    ("results.png",          "Training curves (loss, mAP, P, R)"),
    ("confusion_matrix.png", "Confusion matrix (val set)"),
    ("PR_curve.png",         "Precision-Recall curve"),
    ("F1_curve.png",         "F1 vs confidence threshold"),
]

for filename, title in plots:
    path = f"{RUN_DIR}/{filename}"
    if os.path.exists(path):
        print(f"\n── {title} ──")
        display(Image(path, width=900))
    else:
        print(f"[skip] {filename} not found")

In [ ]:
# ── STEP 11: Sample Predictions ───────────────────────────────────────────────
# Inference pada gambar test acak dan tampilkan hasilnya.
# Hasil disimpan ke Drive di folder predictions/.

import os, glob, random
from IPython.display import Image, display
from ultralytics import YOLO

RUN_NAME     = "ship_corrosion_640_yolov8l_tuned"   # ganti ke ship_corrosion_640_yolov8l jika pakai STEP 7
OUTPUT_DIR   = "/content/drive/MyDrive/colab/vessel-corrosion-runs"
BEST_PT      = f"{OUTPUT_DIR}/detect/{RUN_NAME}/weights/best.pt"
TEST_DIR     = "data/yolo_640/images/test"
SAMPLE_COUNT = 6

model = YOLO(BEST_PT)

test_images = glob.glob(f"{TEST_DIR}/*.jpg") + glob.glob(f"{TEST_DIR}/*.png")
samples     = random.sample(test_images, min(SAMPLE_COUNT, len(test_images)))

results = model.predict(
    source   = samples,
    imgsz    = 640,
    conf     = 0.25,
    iou      = 0.45,
    save     = True,
    project  = f"{OUTPUT_DIR}/predictions",
    name     = RUN_NAME,
    exist_ok = True,
    device   = 0,
)

saved_imgs = sorted(
    glob.glob(f"{OUTPUT_DIR}/predictions/{RUN_NAME}/*.jpg") +
    glob.glob(f"{OUTPUT_DIR}/predictions/{RUN_NAME}/*.png")
)
for path in saved_imgs[:SAMPLE_COUNT]:
    print(os.path.basename(path))
    display(Image(path, width=640))